# Large-CFO front-end boundary

This notebook slows down one bounded SDR claim: **the old `\pi/4` alias note was a symbol-rate statement, so the honest CFO window changes when the estimator sees a different sample rate.**

The goal is not to build a full modem. It is to make one scope boundary impossible to miss.


## 1. Derivation in one line

For QPSK 4th-power coarse recovery, the per-sample phase increment must satisfy `|\omega| < \pi/4`.
With `\omega = 2\pi \Delta f / F_s`, that means

`|\Delta f| < F_s / 8`.

If `F_s = L R_s`, then the normalized coarse window is

`|\Delta f| / R_s < L / 8`.

So the symbol-rate note is the special case `L = 1`.


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if not (repo / 'scripts').exists():
    repo = repo.parent
sys.path.insert(0, str(repo / 'scripts'))

from large_cfo_front_end import alias_limit_normalized, sweep_normalized_cfo

sps_values = [1, 2, 4]
normalized_cfos = [round(step * 0.05, 2) for step in range(0, 11)]
rows = sweep_normalized_cfo(sps_values, normalized_cfos, symbol_count=192, noise_std=0.012, seed=19)
rows[:3]


## 2. Honest windows by samples per symbol

These are the clean theoretical limits for the same 4th-power idea at three observation rates.


In [ ]:
[(sps, alias_limit_normalized(sps)) for sps in sps_values]


## 3. One offset worth reading literally

A good checkpoint is `\Delta f / R_s = 0.30`. It sits far outside the `1 sps` window, outside the `2 sps` window, and still inside the `4 sps` window.

That makes the same physical CFO land in three different interpretive regimes.


In [ ]:
focus = [row for row in rows if abs(row.normalized_cfo - 0.30) < 1e-9]
[(row.samples_per_symbol, round(row.estimated_normalized_cfo, 4), row.honest) for row in focus]


## 4. What this notebook is actually checking

The waveform here is a timed QPSK hold model. That is deliberate.

It is enough to test the **sample-rate scaling law** without pretending we already modeled pulse shaping, matched filtering, or a complete receive chain.

So this notebook is a boundary card, not a final architecture argument.


## 5. Problems worth pushing next

1. Replace the hold model with a pulse-shaped waveform and ask whether the same range picture still reads cleanly after matched filtering.
2. Compare an oversampled 4th-power front end against a band-edge FLL so the repo can say when each one is the cleaner object.
3. Add one pilot-aided comparison only if it sharpens the branch decision instead of widening the topic for its own sake.

Useful companions:

- `notes/when-symbol-rate-carrier-recovery-stops-being-enough.md`
- `assets/2026-05-19-large-cfo-front-end-boundary.png`
- `assets/2026-05-19-large-cfo-front-end-boundary.csv`
- `notes/2026-05-18-large-cfo-front-end-research.md`
